[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Electromagnetismo_y_Plasmas_%28EM%29/Fisica_de_Plasmas_%28physics.plasm-ph%29/PLA-05_Simulacion_Gyromotion_Tokamak.ipynb)

# [PLA-05] Confinamiento en Plasma: Toroides Magnéticos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import odeint

# Integración Numérica de la Ecuación de Movimiento para un Ión en un Tokamak
# Fuerza de Lorentz: F = q * (v x B)
# Un campo magnético puramente toroidal B_phi causa un gradiente, induciendo una deriva vertical.

q = 1.0 # Carga de la partícula
m = 1.0 # Masa

# Campo magnético toroidal simple: B_phi = B0 / R
# En coordenadas cilíndricas: Bx = -B0*y/R^2, By = B0*x/R^2, Bz = 0
def B_field(pos):
    x, y, z = pos
    R2 = x**2 + y**2
    if R2 == 0: return np.array([0,0,0])
    B0 = 10.0
    return np.array([-B0 * y / R2, B0 * x / R2, 0])

def lorentz_force(state, t):
    pos = np.array(state[0:3])
    v = np.array(state[3:6])
    B = B_field(pos)
    a = (q/m) * np.cross(v, B)
    return [v[0], v[1], v[2], a[0], a[1], a[2]]

# Estado inicial: R=2, ligeramente desplazado. Velocidad inicial perpendicular y paralela.
initial_state = [2.0, 0.0, 0.0,  0.5, 2.0, 0.5]
time = np.linspace(0, 15, 2000)

sol = odeint(lorentz_force, initial_state, time)
X, Y, Z = sol[:, 0], sol[:, 1], sol[:, 2]

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Graficar la trayectoria del Ión
ax.plot(X, Y, Z, lw=1.5, color='orange', label='Trayectoria Iónica')

# Pintar un toro de referencia (Centro del Plasma)
theta = np.linspace(0, 2*np.pi, 50)
phi = np.linspace(0, 2*np.pi, 50)
THETA, PHI = np.meshgrid(theta, phi)
R_torus, r_torus = 2.0, 0.5
Xt = (R_torus + r_torus*np.cos(THETA)) * np.cos(PHI)
Yt = (R_torus + r_torus*np.cos(THETA)) * np.sin(PHI)
Zt = r_torus * np.sin(THETA)
ax.plot_wireframe(Xt, Yt, Zt, color='blue', alpha=0.1)

ax.set_title("Física de Plasmas: Confinamiento Magnético Toroidal")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.legend()
plt.show()
# FÍSICA: Se puede apreciar el rápido movimiento circular (Cyclotron) 
# acoplado al movimiento largo (Parallel) a lo largo del toro.
# Además se produce lentamente una deriva en Z debido al gradiente magnético (delft B).

